# Clase 4 — Sistemas de referencia y proyecciones cartográficas

**Sistemas de Información Geográfica**
Especialización en Ciencias Sociales Computacionales — Universidad Nacional Guillermo Brown

| | |
|---|---|
| **Unidad del programa** | 4 — Sistemas de referencia y proyecciones cartográficas |
| **Duración** | 3 horas |
| **Versión** | 2026.1 |
| **Docente** | Renzo Polo |
| **Licencia** | CC BY-SA 4.0 |

---

## 1. La pregunta de hoy

> ### ¿Cuántos km² tiene el Chaco? ¿Y por qué el mapa del mundo que vimos en la escuela miente?

La Clase 3 terminó con tres preguntas abiertas: **¿por qué EPSG:4326 y no otro? ¿Por qué
para medir superficies hubo que convertir a un tercer sistema? ¿Y qué pasa si uno mide en
el equivocado?**

## 2. Objetivos de esta clase

Al terminar, deberías poder:

1. **Reconocer** que toda proyección deforma superficies, formas y/o distancias.
2. **Distinguir** `set_crs()` de `to_crs()`.
3. **Elegir** el sistema adecuado para nuestras visualizaciones y análisis espaciales.

## 3. Material de esta clase

- **Presentación Clase 4**, diapositivas 1–23.

| Bloque de la notebook | Diapositivas |
|---|---|
| 5 — La Tierra no entra en una hoja | 13–17 y 23 |
| 6 — Declarar y transformar: `set_crs` y `to_crs` | 20–22 |
| 7 — Los sistemas que usa este curso | 18–19 |


## 4. Preparación del entorno

In [ ]:
!pip install -q "geopandas==1.0.1" "matplotlib==3.9.2"
!wget -q -O sig_utils.py https://raw.githubusercontent.com/renzoepolo/sig-ciencias-sociales/main/sig_utils.py

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from sig_utils import chequear_crs, CRS_ARGENTINA

# Los datos del curso viven en un repositorio público de GitHub.
DATOS = "https://raw.githubusercontent.com/renzoepolo/sig-ciencias-sociales/main/datos/"

print(f"GeoPandas {gpd.__version__} — entorno listo")

---

## 5. La Tierra no entra en una hoja

> 📽️ **Presentación Clase 4, diapositivas 13–17 y 23**.

La Tierra es un cuerpo de tres dimensiones y el mapa es una hoja de dos. Pasar de una a la
otra **siempre** deforma las superficies, las formas o las distancias. No existe
la proyección sin deformación, y por eso no existe el mapa "correcto": existe el mapa
adecuado para lo que uno quiere mostrar.

Para *ver* esa deformación vamos a usar tres capas:

| Capa | Qué es |
|---|---|
| `paises` | Que ya conocemos de la Clase 1 |
| `canevas` | Red de paralelos y meridianos, cada 15° |
| `indicatriz` | **Indicatrices de Tissot**: círculos de 500 km de radio, todos iguales sobre la Tierra |

En las indicatrices **todos los círculos son idénticos**, cualquier diferencia de tamaño o de forma que se vea en el mapa son producto de la transformación.

In [ ]:
paises = gpd.read_file(DATOS + "paises.gpkg")
canevas = gpd.read_file(DATOS + "canevas.gpkg")
indicatriz = gpd.read_file(DATOS + "indicatriz_tissot.gpkg")

# La Antártida y los casquetes polares se van al infinito en Mercator, así que
# los dejamos afuera para poder comparar los mapas entre sí. clip() recorta las
# geometrías por un rectángulo: oeste, sur, este, norte.
paises = paises[paises["nombre"] != "Antarctica"]
canevas = canevas.clip((-180, -84, 180, 84))

print("Sistema de coordenadas de las tres capas:", paises.crs.name)
print(f"{len(indicatriz)} indicatrices de {indicatriz['radio_km'][0]:.0f} km de radio")

### 👀 El mundo tal como viene: EPSG:4326

Empecemos por lo más simple, los países solos, sin tocar nada.

In [ ]:
fig, eje = plt.subplots(figsize=(11, 6))
paises.plot(ax=eje, color="#d0ddb0", edgecolor="#809848", linewidth=0.3)
eje.set_title("El mundo en EPSG:4326 — coordenadas geográficas")
eje.set_axis_off()
plt.show()

### ▶️ Le agregamos el canevás

Los paralelos y meridianos. En este sistema son todos rectas perpendiculares entre sí, y
los cuadrados de la grilla salen todos iguales.

In [ ]:
fig, eje = plt.subplots(figsize=(11, 6))
paises.plot(ax=eje, color="#d0ddb0", edgecolor="#809848", linewidth=0.3)
canevas.plot(ax=eje, color="#b0b0b0", linewidth=0.4)
eje.set_title("El mundo en EPSG:4326, con paralelos y meridianos")
eje.set_axis_off()
plt.show()

### ▶️ Y ahora las indicatrices de Tissot

In [ ]:
fig, eje = plt.subplots(figsize=(11, 6))
paises.plot(ax=eje, color="#d0ddb0", edgecolor="#809848", linewidth=0.3)
canevas.plot(ax=eje, color="#b0b0b0", linewidth=0.4)
indicatriz.boundary.plot(ax=eje, color="#c0392b", linewidth=0.9)
eje.set_title("EPSG:4326 — todos estos círculos miden 500 km de radio")
eje.set_axis_off()
plt.show()

### 🔍 Ya en el primer mapa hay deformación

Cerca del Ecuador los círculos son círculos. Hacia los polos se achatan: quedan elipses
apoyadas, más anchas que altas. Es decir que **EPSG:4326 ya es una proyección**, y no una
muy buena: estira las distancias este-oeste a medida que uno se aleja del Ecuador.

Esto importa porque EPSG:4326 es lo que viene por defecto en casi todo dato que uno baja,
y es muy fácil quedarse ahí y medir sobre él sin darse cuenta.

### ▶️ La misma Tierra, otra proyección: Mercator (EPSG:3857)

Para no repetir tres líneas de dibujo en cada mapa, las guardamos en una función. Es la
misma receta de recién: países, canevás, indicatrices; lo único que cambia es a qué sistema
se reproyectan las tres capas antes de dibujarlas.

In [ ]:
def mapa_mundi(crs, titulo):
    fig, eje = plt.subplots(figsize=(11, 6))
    paises.to_crs(crs).plot(ax=eje, color="#d0ddb0", edgecolor="#809848", linewidth=0.3)
    canevas.to_crs(crs).plot(ax=eje, color="#b0b0b0", linewidth=0.4)
    indicatriz.to_crs(crs).boundary.plot(ax=eje, color="#c0392b", linewidth=0.9)
    eje.set_title(titulo)
    eje.set_axis_off()
    plt.show()


mapa_mundi("EPSG:3857", "Mercator (EPSG:3857) — el de Google Maps")

Mercator es una proyección **conforme**: conserva las formas, y por eso los círculos siguen
siendo círculos. Lo que no conserva es la superficie, y se ve enseguida: los círculos del
norte son enormes comparados con los del Ecuador, aunque en el planeta midan exactamente
lo mismo.

### ▶️ Robinson, una proyección de compromiso

In [ ]:
mapa_mundi("ESRI:54030", "Robinson — no conserva nada, pero no deforma mucho")

### ▶️ Mollweide, una proyección equivalente

In [ ]:
mapa_mundi("ESRI:54009", "Mollweide — conserva las superficies")

Mollweide es **equivalente**: cada círculo encierra la misma superficie que los demás,
aunque hacia los bordes se incline y se deforme.

### ✅ Comprobación — Groenlandia contra Argentina

Hasta acá miramos. Ahora midamos, que es donde la diferencia deja de ser estética. Tomamos
dos países y calculamos su superficie dos veces: en Mercator y en Mollweide.

In [ ]:
dos_paises = paises[paises["nombre_es"].isin(["Groenlandia", "Argentina"])]

for crs, nombre_proyeccion in [("EPSG:3857", "Mercator"), ("ESRI:54009", "Mollweide")]:
    medido = dos_paises.to_crs(crs)
    superficies = medido.geometry.area / 1_000_000          # de m² a km²

    groenlandia = superficies[medido["nombre_es"] == "Groenlandia"].iloc[0]
    argentina = superficies[medido["nombre_es"] == "Argentina"].iloc[0]

    print(f"{nombre_proyeccion:10}  Groenlandia {groenlandia:12,.0f} km²   "
          f"Argentina {argentina:12,.0f} km²   "
          f"Groenlandia es {groenlandia / argentina:.1f} veces Argentina")

### 🔍 Interpretación

Según Mercator, Groenlandia es **ocho veces** Argentina. Según Mollweide, es **más chica**
que Argentina. Los dos números salen del mismo origen y del mismo código, la diferencia es producto de la proyección.

El valor correcto es el de Mollweide, porque es la proyección equivalente. Groenlandia
mide unos 2,2 millones de km² y Argentina unos 2,8 millones. Mercator no está diseñado con el objetivo de medir superficies, sino de conservar las formas y
los ángulos, ya que fue concebida para navegación.

Mercator es el mapamundi que casi todos tenemos en la cabeza, porque es el del aula y el
de las aplicaciones web. Sistemáticamente agranda lo que está lejos del Ecuador (Europa,
Estados Unidos, Rusia) y achica lo que está cerca (África, el sudeste asiático, América
Central). Para un científico social esto no es una curiosidad técnica: es un ejemplo de
manual de cómo una decisión técnica, tomada en el siglo XVI por razones de navegación,
termina moldeando la representación del mundo de todos.

---

## 6. Declarar y transformar: `set_crs` y `to_crs`

> 📽️ **Presentación Clase 4, diapositivas 20–22**.

Son dos operaciones distintas y se confunden todo el tiempo:

| Método | Qué hace | Analogía |
|---|---|---|
| `set_crs()` | **Declara** en qué sistema están las coordenadas. No toca ni un número. | Decir que esos 20 grados son Celsius y no Fahrenheit |
| `to_crs()` | **Transforma** las coordenadas a otro sistema. Recalcula cada vértice. | Convertir esos 20 °C a 68 °F |

Vamos a verlo con el padrón de escuelas de la Clase 1, una tabla con dos columnas de números.

In [ ]:
escuelas = gpd.read_file(DATOS + "escuelas_primarias.gpkg")

# Nos quedamos con lo que tendría un CSV bajado de un portal: nombre y dos números.
padron = pd.DataFrame({
    "establecimiento": escuelas["establecimiento"],
    "longitud": escuelas["longitud"],
    "latitud": escuelas["latitud"],
})
padron.head()

### ▶️ De dos columnas a una capa

`points_from_xy()` arma un punto con cada par de números. El código es extraído de la Clase 1 pero se eliminó/comentó adrede el parámetro: `crs="EPSG:4326"`. **No le decimos en qué sistema están**.

In [ ]:
puntos = gpd.GeoDataFrame(
    padron,
    geometry=gpd.points_from_xy(padron["longitud"], padron["latitud"]),
    # crs="EPSG:4326"
)

print("Sistema de coordenadas:", puntos.crs)

### 👀 Y sin embargo el mapa sale perfecto

In [ ]:
fig, eje = plt.subplots(figsize=(6, 9))
puntos.plot(ax=eje, color="#c0392b", markersize=0.5)
eje.set_title(f"{len(puntos):,} escuelas primarias, sin sistema declarado")
eje.set_axis_off()
plt.show()

Se ve Argentina. Pero el mapa no prueba nada: matplotlib dibujó pares de números en un
plano cualquiera. Nadie dijo qué son esos números.

### ▶️ El error que hay que conocer

Intentemos pasarlos a un sistema métrico para poder medir. **La celda que sigue falla a
propósito**: leé el mensaje de error, es el que te vas a encontrar el resto del curso.

In [ ]:
puntos.to_crs("EPSG:5347")

### 🔍 Por qué falla (Diapositiva 22)

El mensaje dice que no se puede transformar una capa que **no tiene CRS declarado**. Es
razonable: transformar es calcular *desde* un sistema *hacia* otro, y acá falta el punto de
partida. GeoPandas no lo adivina y lo advierte.

### ▶️ Primero declarar, después transformar

El valor no se inventa: sale de la documentación de la fuente. El padrón del Ministerio de
Educación publica en coordenadas geográficas WGS 84, es decir **EPSG:4326**.

In [ ]:
puntos = puntos.set_crs("EPSG:4326")      # declarar: no mueve nada
print("Después de set_crs: ", puntos.crs.name)
print("Primer punto:       ", puntos.geometry.iloc[0])

In [ ]:
puntos_metros = puntos.to_crs("EPSG:5347")   # transformar: recalcula todo
print("Después de to_crs:  ", puntos_metros.crs.name)
print("Primer punto:       ", puntos_metros.geometry.iloc[0])

### ✅ Comprobación — ¿cuál de las dos movió las coordenadas?

Comparemos las tres versiones del mismo punto: el original sin declarar, el declarado y el
transformado.

In [ ]:
original = gpd.points_from_xy(padron["longitud"], padron["latitud"])[0]

print(f"Sin declarar   {original.x:14.4f} {original.y:14.4f}   (¿grados? ¿metros? no se sabe)")
print(f"set_crs 4326   {puntos.geometry.iloc[0].x:14.4f} {puntos.geometry.iloc[0].y:14.4f}   grados")
print(f"to_crs 5347    {puntos_metros.geometry.iloc[0].x:14.4f} {puntos_metros.geometry.iloc[0].y:14.4f}   metros")

### 🔍 Interpretación

`set_crs()` dejó los números **idénticos**: lo único que cambió es que ahora la capa sabe
que son grados. `to_crs()` sí los cambió, y bastante: pasaron a ser metros medidos desde el
meridiano central de la faja 5.

La regla práctica, y es la única que hay que recordar:

> **`set_crs()`** se usa una sola vez, cuando el dato llegó sin metadato.
> 
> **`to_crs()`** se usa cada vez que hace falta medir.
> 
> Si usás `set_crs()` sobre una capa que ya tenía CRS estás asignándole un nuevo sistema de coordenadas a tu GeoDataFrame.

---

## 7. Los sistemas que usa este curso

> 📽️ **Presentación Clase 4, diapositivas 18–19** — El sistema Gauss-Krüger y el
> identificador de referencia espacial (SRID).

Un **SRID** es el número con que se identifica un sistema de coordenadas. El catálogo más
usado es el de la **EPSG**, y por eso los códigos se escriben `EPSG:4326`, `EPSG:5347`. Se
buscan en <https://epsg.org/search/by-name>.

En Argentina, el sistema oficial de coordenadas planas es **Gauss-Krüger POSGAR 2007**. El país se divide en **siete fajas** de 3° de ancho, y cada una
tiene su propio código EPSG. La faja se elige por la longitud del lugar, y dentro de ella la deformación es despreciable.

| Faja | Código EPSG | Cubre desde | Hasta | Meridiano central |
|---|---|---|---|---|
| 1 | EPSG:5343 | 73,5°O | 70,5°O | 72°O |
| 2 | EPSG:5344 | 70,5°O | 67,5°O | 69°O |
| 3 | EPSG:5345 | 67,5°O | 64,5°O | 66°O |
| 4 | EPSG:5346 | 64,5°O | 61,5°O | 63°O |
| 5 | EPSG:5347 | 61,5°O | 58,5°O | 60°O |
| 6 | EPSG:5348 | 58,5°O | 55,5°O | 57°O |
| 7 | EPSG:5349 | 55,5°O | 52,5°O | 54°O |

In [ ]:
for nombre, codigo in CRS_ARGENTINA.items():
    print(f"{nombre:26} {codigo}")

### 👀 La misma Argentina, cuatro sistemas

Antes de medir, mirémosla. Las cuatro son el mismo país y el mismo archivo.

In [ ]:
provincias = gpd.read_file(DATOS + "provincias_arg.gpkg")

sistemas = [
    ("EPSG:4326", "EPSG:4326 — geográficas, en grados"),
    ("EPSG:3857", "EPSG:3857 — Web Mercator"),
    ("EPSG:5347", "EPSG:5347 — POSGAR 2007, faja 5"),
    ("ESRI:102033", "ESRI:102033 — Albers, equivalente"),
]

fig, ejes = plt.subplots(1, 4, figsize=(14, 7))
for (crs, titulo), eje in zip(sistemas, ejes):
    provincias.to_crs(crs).boundary.plot(ax=eje, color="#555555", linewidth=0.5)
    eje.set_title(titulo, fontsize=8)
    eje.set_axis_off()
plt.show()

Los cuatro dibujos son parecidos, y esa es justamente la trampa: **mirando el mapa no se
distingue cuál sirve para medir**. Hay que medir para darse cuenta.

### ▶️ Midamos el Chaco

Para elegir la faja hay que saber por qué longitudes pasa la provincia. El propio archivo
de provincias trae además la superficie oficial, con lo cual tenemos contra qué comparar.

In [ ]:
chaco = provincias[provincias["provincia"] == "Chaco"]

superficie_oficial = chaco["superficie_km2"].iloc[0]

# total_bounds devuelve los cuatro extremos de la capa: oeste, sur, este, norte.
oeste, sur, este, norte = chaco.total_bounds

print(f"Superficie oficial del Chaco: {superficie_oficial:,.0f} km²")
print(f"La provincia va de {oeste:.1f}° a {este:.1f}° de longitud")
print(f"Su punto medio está en {(oeste + este) / 2:.1f}°")

El punto medio del Chaco cae dentro de la **faja 5** (EPSG:5347), que cubre de 61,5°O a
58,5°O, así que esa es la que le corresponde. Ahora bien, la provincia arranca en los 63,4°O:
buena parte de su mitad oeste queda fuera de la faja. Es lo habitual, una provincia entera
casi nunca entra en una sola faja.

### ✅ Comprobación — la misma provincia en cuatro sistemas

In [ ]:
for crs, etiqueta in sistemas:
    medida = chaco.to_crs(crs).geometry.area.iloc[0]

    if crs == "EPSG:4326":
        print(f"{etiqueta:38} {medida:12,.1f}  grados cuadrados (no comparable)")
    else:
        km2 = medida / 1_000_000
        print(f"{etiqueta:38} {km2:12,.0f} km²   {km2 / superficie_oficial * 100:6.1f} % del oficial")

### 🔍 Interpretación

- **EPSG:4326** devuelve un número sin unidad interpretable y un mensaje de alerta. Un grado de longitud mide unos
  111 km en el Ecuador y cero en el polo, así que un "grado cuadrado" no es una superficie.
  Es el error que cometimos sin querer en la Clase 3, y fijate que esta vez **GeoPandas
  avisa**: la advertencia *"Geometry is in a geographic CRS"* que aparece arriba del
  resultado es exactamente esto.
- **EPSG:3857** devuelve kilómetros cuadrados de verdad, pero **un 25 % de más**. Es el peor
  caso posible: el número parece bien, tiene la unidad correcta y está mal. Web Mercator
  sirve para dibujar mapas web y para nada más.
- **EPSG:5347**, la faja que le corresponde, da prácticamente el valor oficial.
- **ESRI:102033**, la proyección equivalente de Sudamérica, también, y además funciona para
  todo el país a la vez.

De acá salen las dos reglas que el curso va a seguir hasta el final:

> **Si queremos medir superficies en todo el país y compararlas: ESRI:102033.**
> 
> **Para medir distancias o hacer buffers en una zona acotada: la faja POSGAR que le toca.**
> 
> **Nunca EPSG:3857 para medir. Nunca EPSG:4326 para medir.**

---

## 8. Cierre

### Lo que vimos

Un sistema de referencia de coordenadas es **una convención**. Como todo modelo, simplifica, y al simplificar deforma.
La pregunta nunca es si el mapa deforma sino **qué deformó y si eso
influye en mis representaciones o análisis, y por ende, en mis conclusiones**.

| Situación | Qué hacer |
|---|---|
| El dato llegó sin CRS | `set_crs()` con el valor que dice la fuente |
| El dato llegó con CRS y quiero medir | `to_crs()` al sistema adecuado |
| Quiero comparar superficies en el mundo | Una proyección equivalente (Mollweide) |
| Quiero medir en Argentina | ESRI:102033, o la faja POSGAR de la zona |
| Quiero un mapa web de fondo | EPSG:3857, para el fondo |

### Glosario de la clase

| Término | Definición |
|---|---|
| **Elipsoide** | Modelo matemático de la forma de la Tierra, achatado en los polos. |
| **Geoide** | Modelo físico, definido por mediciones de gravedad. Más exacto y más incómodo. |
| **Datum** | El elipsoide elegido más su posición respecto de la Tierra. WGS 84 y POSGAR 2007 son datums. |
| **Proyección** | La transformación matemática que lleva la superficie curva al plano. |
| **Conforme** | Proyección que conserva las formas y los ángulos. Mercator. |
| **Equivalente** | Proyección que conserva las superficies. Mollweide, Albers. |
| **SRID / EPSG** | El código con que se identifica un sistema de coordenadas. |
| **Gauss-Krüger** | Esquema de fajas angostas; en Argentina, las siete fajas de POSGAR 2007. |
| **`set_crs()`** | Declara el sistema que la capa ya tenía. No mueve coordenadas. |
| **`to_crs()`** | Convierte las coordenadas a otro sistema. Las recalcula todas. |

### La próxima clase

Con los datos ya en su sistema correcto, empezamos a hacer mapas en serio: cuál es el mapa
adecuado para cada tipo de dato, cómo se clasifica una variable en clases y por qué un mapa
de conteos y un mapa de porcentajes de la misma variable cuentan historias distintas.

> **Clase 5 — ¿Dónde está la pobreza? ¿Y el mapa cambia la respuesta?**

## 9. Bibliografía

- Instituto Geográfico Nacional (2021). *Sistemas de referencia y marcos de referencia
  geodésicos*. <https://www.ign.gob.ar/NuestrasActividades/Geodesia>
- Olaya, V. (2020). *Sistemas de Información Geográfica*, cap. 3: "Fundamentos
  cartográficos y geodésicos". <https://volaya.github.io/libro-sig/>
- Monmonier, M. (1991). *How to Lie with Maps*. University of Chicago Press. Cap. 2,
  "Maps Can't Be Made Without Generalization", y cap. 3, "Blunders That Mislead".
- Snyder, J. P. (1987). *Map Projections: A Working Manual*. USGS Professional Paper 1395.
  <https://pubs.usgs.gov/pp/1395/report.pdf>